In [ ]:
!pip -q uninstall -y numpy spacy thinc scispacy negspacy
!pip -q install "numpy==1.26.4" "spacy==3.7.4" "thinc==8.2.3" scispacy negspacy
!pip -q install https://s3-us-west-2.amazonaws.com/ai2-s2-scispacy/releases/v0.5.4/en_core_sci_md-0.5.4.tar.gz

# Restart kernel so the new numpy binary is actually loaded
import os
os.kill(os.getpid(), 9)

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
jaxlib 0.7.2 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
xarray-einstats 0.10.0 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
opencv-python 4.13.0.92 requires numpy>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
rasterio 1.5.0 requires numpy>=2, but you have numpy 1.26.4 which is incompatible.
jax 0.7.2 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
opencv-contrib-python 4.13.0.92 requires numpy>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
opencv-python-headless 4.13.0.92 requires numpy>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
tobler 0.13.0 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
cupy-cuda12x 14.0.1 requires numpy

In [1]:
cases = [
    {
        "case_id": "CASE_001",
        "note_text": "45-year-old male referred for evaluation of gait disturbance and intermittent tremor. Symptoms began approximately 18 months ago and have slowly progressed. Patient reports mild unsteadiness while walking, occasional hand tremors when reaching for objects, and two falls in the past year without loss of consciousness. No history of seizures, severe headaches, or visual hallucinations. Denies chest pain, palpitations, shortness of breath, or syncopal episodes. Sleep is generally normal; no reports of REM sleep behavior disorder. Appetite and weight are stable, with no recent weight loss or gain. Family history is notable for a maternal uncle with Parkinson’s disease diagnosed in his 60s. No known family history of early-onset neurodegenerative disorders, epilepsy, or muscular dystrophy. Neurologic exam shows mild truncal ataxia and intention tremor on finger-to-nose testing. Muscle strength is 5/5 throughout, with no evidence of spasticity or rigidity. Reflexes are symmetric. Impression: Progressive gait ataxia and action tremor, no evidence of seizures or major cardiopulmonary involvement at this time. Further workup with MRI brain and genetic testing is recommended."
    },
    {
        "case_id": "CASE_002",
        "note_text": "H&P: 32F presents with chronic headaches. Denies photophobia, denies nausea or vomiting. No seizures. No loss of consciousness. ROS otherwise negative for dizziness or weakness. Father had migraines. Pt reports childhood febrile seizures but none since age 3. No current seizure activity."
    },
    {
        "case_id": "CASE_003",
        "note_text": "ED NOTE: Pt brought in for altered mental status. On arrival, no witnessed seizure activity. No incontinence. Denies headache. No neck stiffness. ROS negative for chest pain. Sister has epilepsy. No tremors noted. No ataxia. Cannot rule out underlying metabolic cause."
    },
    {
        "case_id": "CASE_004",
        "note_text": "PROGRESS NOTE: 58 yo M w/ hx DM2. Denies SOB, denies CP. No seizure-like episodes. No visual hallucinations. Neuro exam: no focal deficits, no ataxia. Review of systems negative for weakness or numbness. Mother denies any family history of neurologic disorders."
    },
    {
        "case_id": "CASE_005",
        "note_text": "ADMISSION NOTE: 21F evaluated for possible seizure. Patient states she has never had seizures. No aura. No tongue biting. No urinary incontinence. No head injury. ROS negative for palpitations, dizziness, syncope. Uncle w/ hx epilepsy. No current neurologic deficits."
    },
    {
        "case_id": "CASE_006",
        "note_text": "CLINIC VISIT: Pt here for tremor evaluation. Denies seizures but reports intermittent shaking of hands. No loss of consciousness. No confusion post-event. No ataxia. No visual disturbances. ROS otherwise unremarkable. Family history negative for seizure disorders."
    },
    {
        "case_id": "CASE_007",
        "note_text": "ICU NOTE: Pt intubated. No seizure activity observed overnight. EEG without epileptiform discharges. No fevers. No focal deficits appreciated. No prior seizure history per family. Wife states patient never had seizures. Cannot completely exclude subclinical seizure."
    },
    {
        "case_id": "CASE_008",
        "note_text": "NEURO CONSULT: 34F w/ dizziness. No clear seizure episodes. Denies convulsions. No tongue biting. No incontinence. No postictal confusion. ROS negative for weakness, numbness, vision loss. Brother has seizure disorder. No abnormal movements noted."
    },
    {
        "case_id": "CASE_009",
        "note_text": "HOSPITAL COURSE: Pt admitted after fall. Denies LOC. No seizure-like activity reported. No urinary incontinence. No witnessed convulsions. No prior neurologic disease. Mother with Parkinson's disease. ROS negative for chest pain or dyspnea."
    },
    {
        "case_id": "CASE_010",
        "note_text": "FOLLOW-UP NOTE:Patient evaluated for recurrent syncope. No seizures observed during telemetry. No prior history of epilepsy. Denies aura, denies tonic-clonic activity. No focal weakness. ROS negative for dizziness today. Had seizures as infant per mother, none in adulthood."
    },
]

print("Loaded cases:", len(cases))

Loaded cases: 10


In [2]:
import spacy
from negspacy.negation import Negex
from negspacy.termsets import termset

nlp = spacy.load("en_core_sci_md")

ts = termset("en_clinical")

nlp.add_pipe(
    "negex",
    config={"neg_termset": ts.get_patterns()}
)

print(nlp.pipe_names)

/usr/local/lib/python3.12/dist-packages/spacy/language.py:2195: FutureWarning: Possible set union at position 6328
  deserializers["tokenizer"] = lambda p: self.tokenizer.from_disk(  # type: ignore[union-attr]


['tok2vec', 'tagger', 'attribute_ruler', 'lemmatizer', 'parser', 'ner', 'negex']


In [3]:
FAMILY_WORDS = {
    "mother","father","sister","brother","uncle","aunt","wife","husband",
    "son","daughter","maternal","paternal","family history","fhx","per family"
}

def is_family_sentence(sent_text: str) -> bool:
    s = sent_text.lower()
    return any(w in s for w in FAMILY_WORDS)

In [4]:
import re

# --- 1) cues we want to capture ---
CUE_PATTERNS = [
    r"\bdenies\b",
    r"\bdenied\b",
    r"\bno\b",
    r"\bwithout\b",
    r"\bnegative for\b",
    r"\bnever\b",
    r"\bfree of\b",
    r"\babsence of\b",
    r"\black of\b",
    r"\bnot\b",
    r"\bno history of\b",
    r"\bno evidence of\b",
]

# --- 2) phrases that indicate UNCERTAINTY (NOT true exclusion) ---
UNCERTAINTY_PATTERNS = [
    r"\bcannot rule out\b",
    r"\bcan not rule out\b",
    r"\bcan't rule out\b",
    r"\bcannot exclude\b",
    r"\bcan not exclude\b",
    r"\bcan't exclude\b",
    r"\bnot ruled out\b",
    r"\bcannot completely exclude\b",
]

# --- 3) common junk mentions produced by SciSpacy in clinical notes ---
JUNK_SURFACES = {
    "severe", "evidence", "time", "involvement", "cause", "metabolic",
    "activity", "overnight", "abnormal", "movements", "post-event",
}

# optional: drop modifiers like "post-event", "pre-op", etc.
JUNK_PREFIXES = ("post-", "pre-", "peri-", "intra-")

def _find_negation_cue(sentence: str, mention_start_in_sentence: int):
    """Return the closest negation cue that appears BEFORE the mention in the sentence."""
    s = sentence.lower()
    best = None
    best_end = -1

    for pat in CUE_PATTERNS:
        for m in re.finditer(pat, s):
            if m.end() <= mention_start_in_sentence and m.end() > best_end:
                best = m.group(0)
                best_end = m.end()

    return best

def _has_uncertainty(sentence: str):
    s = sentence.lower()
    return any(re.search(p, s) for p in UNCERTAINTY_PATTERNS)

def _clean_surface(surface: str):
    """Normalize surfaces like 'no ataxia' -> ('ataxia', 'no')"""
    s = surface.strip()
    low = s.lower()

    # If the entity span includes the cue itself, normalize it out
    if low.startswith("no "):
        return s[3:].strip(), "no"
    if low.startswith("denies "):
        return s[7:].strip(), "denies"
    if low.startswith("without "):
        return s[8:].strip(), "without"

    return s, None

def _is_good_surface(surface: str):
    low = surface.lower().strip()

    # junk exact matches
    if low in JUNK_SURFACES:
        return False

    # junk prefix patterns like post-event
    if any(low.startswith(p) for p in JUNK_PREFIXES):
        return False

    # too short + not a common clinical abbrev
    if len(low) <= 2 and low not in {"cp", "sob", "loc"}:
        return False

    # avoid single generic words that commonly appear as fragments
    if low in {"major", "recent", "noted", "not", "during"}:
        return False

    return True

def extract_exclusions(note_text: str):
    doc = nlp(note_text)
    excluded = []

    for ent in doc.ents:
        if not getattr(ent._, "negex", False):
            continue

        sent = ent.sent.text.strip()

        # skip family history sentences
        if is_family_sentence(sent):
            continue

        # skip uncertain statements (not true "excluded mention")
        if _has_uncertainty(sent):
            continue

        # clean surface and possibly recover cue from surface itself
        cleaned_surface, cue_from_surface = _clean_surface(ent.text)

        if not _is_good_surface(cleaned_surface):
            continue

        # compute mention start relative to sentence (so cue search is correct)
        sent_start_char = ent.sent.start_char
        mention_start_in_sentence = ent.start_char - sent_start_char

        cue = cue_from_surface or _find_negation_cue(sent, mention_start_in_sentence)

        excluded.append({
            "surface": cleaned_surface,
            "negation_cue": cue,
            "context": "patient",
            "evidence": {
                "snippet": sent,
                "span": [ent.start_char, ent.end_char]
            }
        })

    # dedupe by (surface, span_start, span_end)
    seen = set()
    dedup = []
    for x in excluded:
        key = (x["surface"].lower(), tuple(x["evidence"]["span"]))
        if key not in seen:
            seen.add(key)
            dedup.append(x)

    # add mention_id
    for i, m in enumerate(dedup, 1):
        m["mention_id"] = f"m{i}"

    return dedup

In [5]:
moduleB_outputs = []

for case in cases:
    ex = extract_exclusions(case["note_text"])
    moduleB_outputs.append({
        "case_id": case["case_id"],
        "excluded_mentions": ex
    })
    print(case["case_id"], "->", [m["surface"] for m in ex][:12])

CASE_001 -> ['loss of consciousness', 'seizures', 'headaches', 'visual hallucinations', 'palpitations', 'shortness of breath', 'syncopal episodes', 'REM sleep behavior disorder', 'weight loss', 'spasticity', 'rigidity', 'seizures']
CASE_002 -> ['nausea or vomiting', 'seizures', 'dizziness', 'weakness']
CASE_003 -> ['seizure', 'chest pain', 'tremors']
CASE_004 -> ['SOB', 'CP', 'seizure-like episodes', 'visual hallucinations', 'focal deficits', 'ataxia', 'weakness', 'numbness']
CASE_005 -> ['seizures', 'tongue biting', 'urinary incontinence', 'head injury', 'palpitations', 'dizziness', 'syncope', 'neurologic deficits']
CASE_006 -> ['confusion', 'visual disturbances']
CASE_007 -> ['epileptiform discharges', 'focal deficits']
CASE_008 -> ['seizure', 'episodes', 'tongue biting', 'postictal', 'confusion', 'weakness', 'numbness', 'vision loss']
CASE_009 -> ['LOC', 'seizure-like activity', 'urinary incontinence', 'neurologic disease', 'chest pain', 'dyspnea']
CASE_010 -> ['seizures', 'telemetr

In [7]:
import json
with open("moduleB_output.json", "w") as f:
    json.dump(moduleB_outputs, f, indent=2)

print("✅ Saved moduleB_output.json")


✅ Saved moduleB_output.json


In [12]:
final_output = []

for case in moduleB_outputs:
    formatted_mentions = []

    for i, m in enumerate(case["excluded_mentions"], 1):
        formatted_mentions.append({
            "mention_id": f"m{i}",
            "surface": m.get("surface"),
            "negation_cue": m.get("negation_cue"),
            "evidence": m.get("evidence"),
            "context": m.get("context", "patient")
        })

    final_output.append({
        "case_id": case["case_id"],
        "excluded_mentions": formatted_mentions
    })

# Save final JSON
import json
with open("moduleB_output.json", "w") as f:
    json.dump(final_output, f, indent=2)

print("✅ Saved correctly formatted moduleB_output.json")

✅ Saved correctly formatted moduleB_output.json
